# Assessment - Statistical Methods for Causal Inference


## Inference with inverse propensity reweighting

The following code illustrates how to use Pyro abstractions to calculate a causal effect using the inverse probability reweighting approach.

In [ ]:
# comment install commands in/out as required
!pip install pyro-ppl

import pyro
from pyro import sample
import numpy as np
from pyro.distributions import Normal, Categorical
import torch
import numpy as np
import pandas as pd

     |████████████████████████████████| 686kB 5.3MB/s 


We have the following causal DAG.

<center><img src="https://cdn.fs.teachablecdn.com/ADNupMnWyR7kCWRvm76Laz/resize=width:1500/https://www.filepicker.io/api/file/zWQ9LKT8RBCtKlsUsiwr" alt="title" title="Causal DAG" width="300"/></center>
<br>

Z1 and Z2 are continuous, while X and Y are binary. We are interested in the causal effect of X on Y.

To find that causal effect, we would have to adjust to both Z1 and Z2. Since Z1 and Z2 are continuous, adjustment would require double integration. That might be analytically intractable as well as challenging using numerical methods. These challenges motivate the use of inverse probability reweighting.

As we saw in the lecture, we can use the probability mass function of X given Z1 and Z2 ( P(X|Z1, Z2) ) as a propensity score and use that in our reweighting algorithm. Let's assume P(X|Z1, Z2) is as follows.

In [ ]:
def pXgivenZ(x, z1, z2):
  prob_X = torch.tensor([
              [0.15, 0.85],
              [0.90, 0.10]
          ])
  if z1 + z2 <= 0:
    return prob_X[0][x]
  else:
    return prob_X[1][x]

For our propensity score calculation, we'll need the joint probability density function of Z1, Z2, X, and Y. Probabilistic programming will make this easy since the program is a generative model of a joint distribution.

We'll create the following probabilistic program. Here, we'll assume we know the parameters of the model, but in practice we'd probably want to learn them from the data.

In [ ]:
def model():
  z1 = sample('Z1', Normal(0, 1))
  z2 = sample('Z2', Normal(0, 1))

  # [
  #  P(X|z1 + z2 <= 0)
  #  P(X|z1 + z2 >  0)
  # ]
  prob_X = torch.tensor([
              [0.15, 0.85],
              [0.90, 0.10]
          ])
  prob_Y = torch.tensor([ # P(Y|X=0, Z)
                         [[0.068, 0.932],
                          [0.267, 0.733]],
                          # P(Y|X=1, Z)
                         [[0.131, 0.869],
                          [0.313, 0.687]]])
#   print(prob_Y[0][0])

  if z1 + z2 <= 0:
    x = sample('X', Categorical(probs=prob_X[0]))
    y = pyro.sample('Y', Categorical(probs=prob_Y[x][0]))
  else:
    x = sample('X', Categorical(probs=prob_X[1]))
    y = pyro.sample('Y', Categorical(probs=prob_Y[x][1]))
  return x, y, z1, z2

# model()

## Using the Trace

Next, we'll use the program to sample from the joint. For each sample, we'll need the sample probability. For that, we'll need to use a probabilistic programming abstraction called a "trace."

Probabilistic programming generally works by executing the program many times, and then reasoning on the ensemble of program executions, which vary because the program is probabilistic. A program execution is typically called an "execution trace", or just "*trace*." The data structure representing a trace stores the sampled values of the variables in the program, the log of the total probability of those outcomes, as well as other useful items. So the trace corresponds to one sample from the joint probability distribution, and the log-probability value is the log-probability of that particular sample.

Pyro has a class called `Trace` that serves as a trace data structure. We'll use it to generate 1000 samples from the joint.

In [ ]:
samples = []
sample_size = 1000
trace_handler = pyro.poutine.trace(model)
for i in range(sample_size):
    trace = trace_handler.get_trace()
    x = trace.nodes['X']['value']
    y = trace.nodes['Y']['value']
    z1 = trace.nodes['Z1']['value']
    z2 = trace.nodes['Z2']['value']
    p = np.exp(trace.log_prob_sum())
    samples.append({'X': x, 'Y': y, 'Z1': z1, 'Z2': z2, 'p':p})

Now, you have what you need to calculate causal effect using the inverse probability reweighting approach.

1. For E(Y|do(X=1)), filter the samples to those where X = 1.
2. For each of the remaining samples, take the values of Z1 and Z2 (and X=1) to calculate the propensity score.
3. Reweight the sample as the sample probability divided by the propensity score.
4. Use these weights to calculate the weighted mean of Y across the samples (or resample with replacement using these weights and take the regular mean). This will give you an estimate of E(Y|do(X=1))
5. Repeat steps 1-4 for E(Y|do(X=0))

Note that in general for this algorithm, you don't need a probabilistic program; you just need a way of getting samples from the joint (or already have historical data that functions as samples from the joint). You *might* not even need to get the probability of a sample if you can use sample frequencies as proxies for joint probabilities.

In [ ]:
# Convert the samples to a Pandas DataFrame to make data manipulation easier
df = pd.DataFrame(samples)

# Debug the samples
# print(df["X"])
# print(df["X"].sum())
# print("df = %s" % df)

# Calculate the propensity score for all data points using the pXgivenZ function for X=1
df_x_1 = df.copy()
df_x_1["p_score"] = df_x_1.apply(lambda x: pXgivenZ(1, x['Z1'], x['Z2']), axis=1)
# Calculate the weight of each data point as its joint probability times its propensity score
df_x_1["weight"] = df_x_1["p"]/df_x_1["p_score"]
# Filter the original samples to those where X==1
df_given_x_1 = df_x_1[df_x_1["X"] == torch.tensor(1)]
# Compute the weighted mean of Y across the filtered samples
exp_y_1_do_x_1 = (df_given_x_1["Y"]*df_given_x_1["weight"]).sum()/df_given_x_1["weight"].sum()

# Calculate the propensity score for all data points using the pXgivenZ function for X=0
df_x_0 = df.copy()
df_x_0["p_score"] = df_x_0.apply(lambda x: pXgivenZ(0, x['Z1'], x['Z2']), axis=1)
# Calculate the weight of each data point as its joint probability times its propensity score
df_x_0["weight"] = df_x_0["p"]/df_x_0["p_score"]
# Filter the original samples to those where X==0
df_given_x_0 = df_x_0[df_x_0["X"] == torch.tensor(0)]
# Compute the weighted mean of Y across the filtered samples
exp_y_1_do_x_0 = (df_given_x_0["Y"]*df_given_x_0["weight"]).sum()/df_given_x_0["weight"].sum()

## ATE
ate = exp_y_1_do_x_1 - exp_y_1_do_x_0
print("Expected values for interventional distributions: E[y|do(x=1)] = %f, E[y|do(x=0)] = %f, ÀTE = %f" % (exp_y_1_do_x_1, exp_y_1_do_x_0, ate))

## NATE for comparison
naive_exp_y_1_given_x_1 = df_given_x_1["Y"].mean()
naive_exp_y_1_given_x_0 = df_given_x_0["Y"].mean()
nate = naive_exp_y_1_given_x_1 - naive_exp_y_1_given_x_0
print("Expected values for conditional distributions: E[Y|X=1] = %f, E[Y|X=0] = %f, NATE = %f" % (naive_exp_y_1_given_x_1, naive_exp_y_1_given_x_0, nate))

Expected values for interventional distributions: E[y|do(x=1)] = 0.967542, E[y|do(x=0)] = 0.903537, ÀTE = 0.064005
Expected values for conditional distributions: E[Y|X=1] = 0.858388, E[Y|X=0] = 0.765250, NATE = 0.093138
